<a href="https://colab.research.google.com/github/mohiuddin2806-lang/machine-learning-/blob/main/CN7050_week6_u2975262.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install -U openai ipywidgets requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Install zstd and pciutils (for lspci GPU detection) and then install Ollama
!sudo apt-get update && sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,427 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,177 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/restricted amd64 Packages [2,137 kB]
Get:13 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [7

In [3]:
# set up local LLM
import subprocess, time, requests, os

# Launch Ollama in the background; re-running this cell is safe.
if not requests.get("http://127.0.0.1:11434/api/tags", timeout=2).ok if False else False:
    pass
try:
    requests.get("http://127.0.0.1:11434/api/tags", timeout=2).raise_for_status()
    print("Ollama is already running")
except Exception:
    ollama_process = subprocess.Popen(
        ["ollama", "serve"], stdout=open("/tmp/ollama.log", "a"),
        stderr=subprocess.STDOUT
    )
    for _ in range(30):
        try:
            requests.get("http://127.0.0.1:11434/api/tags", timeout=2).raise_for_status()
            print("Ollama server ready")
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("Ollama did not start; inspect /tmp/ollama.log")

Ollama server ready


In [4]:
# Download the open-weight instruction model (once per new Colab runtime).
!ollama pull qwen2.5:3b

In [5]:
from openai import OpenAI

# DEFAULT: free, locally served Qwen model via Ollama
MODEL = "qwen2.5:3b"
client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama"  # Dummy string required by SDK; not a paid key
)

# OPTIONAL: paid OpenAI API (uncomment ALL lines below and comment
# out the local MODEL/client lines above).
# import getpass
# MODEL = "gpt-4o-mini"
# client = OpenAI(api_key=getpass.getpass("Enter OpenAI API key: "))

print("Selected model:", MODEL)

Selected model: qwen2.5:3b


In [6]:
# Smoke test: basic inference before introducing tools
reply = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: Travel agent ready."}],
    temperature=0,
)
print(reply.choices[0].message.content)

Travel agent ready.


In [7]:
from datetime import date
from typing import Dict

BOOKING_LOG = []

def book_flight(passenger_name: str, from_city: str,
                to_city: str, travel_date: str) -> dict:
    """Simulate a booking. No real ticket is purchased."""
    if not all(isinstance(x, str) and x.strip()
               for x in (passenger_name, from_city, to_city, travel_date)):
        return {"status": "error", "message": "All four fields are required."}
    try:
        parsed = date.fromisoformat(travel_date)
    except ValueError:
        return {"status": "error", "message": "Use date format YYYY-MM-DD."}
    if parsed < date.today():
        return {"status": "error", "message": "Travel date must not be in the past."}
    record = {
        "status": "simulated_only",
        "passenger_name": passenger_name.strip(),
        "from_city": from_city.strip(),
        "to_city": to_city.strip(),
        "travel_date": travel_date,
        "message": "SIMULATION ONLY: no flight has actually been booked."
    }
    BOOKING_LOG.append(record)
    return record

# Direct function test (no model/API call)
print(book_flight("Test Student", "London", "Paris", "2030-06-15"))
BOOKING_LOG.clear()  # Keep the example clean

{'status': 'simulated_only', 'passenger_name': 'Test Student', 'from_city': 'London', 'to_city': 'Paris', 'travel_date': '2030-06-15', 'message': 'SIMULATION ONLY: no flight has actually been booked.'}


In [8]:
tools = [{
    "type": "function",
    "function": {
        "name": "book_flight",
        "description": (
            "Simulate a flight booking only after the passenger has supplied "
            "their full name, departure city, destination city and future travel date. "
            "This does NOT make a real reservation."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "passenger_name": {"type": "string", "description": "Passenger full legal name"},
                "from_city": {"type": "string", "description": "Departure city"},
                "to_city": {"type": "string", "description": "Destination city"},
                "travel_date": {"type": "string", "description": "Travel date in YYYY-MM-DD format"},
            },
            "required": ["passenger_name", "from_city", "to_city", "travel_date"],
            "additionalProperties": False,
        },
    },
}]
print("Tool name:", tools[0]["function"]["name"])

Tool name: book_flight


In [9]:
import json

SYSTEM_PROMPT = """You are a friendly travel assistant demonstrating function calling.
Your ONLY booking tool simulates a booking and does not buy a ticket.
Before calling book_flight, obtain all four details from the user:
(1) passenger full legal name, (2) departure city, (3) destination city,
(4) future travel date in YYYY-MM-DD format.
Ask a concise follow-up question for missing details. Do not invent details.
Do not claim a real booking was made. When the tool returns an error, explain
what needs to be corrected. After a successful tool result, clearly say
'SIMULATED booking' and summarize only the returned details.
"""

def new_conversation():
    return [{"role": "system", "content": SYSTEM_PROMPT}]

messages = new_conversation()

TOOL_REGISTRY = {"book_flight": book_flight}

def travel_agent(user_message: str, messages: list, max_rounds: int = 4) -> str:
    """Multi-turn tool-calling loop compatible with local Ollama or OpenAI."""
    messages.append({"role": "user", "content": user_message})
    for step in range(max_rounds):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
            temperature=0,
        )
        assistant = response.choices[0].message
        # Append SDK message as a serializable dictionary.
        messages.append(assistant.model_dump(exclude_none=True))

        if not assistant.tool_calls:
            return assistant.content or "Please provide more details."

        for call in assistant.tool_calls:
            name = call.function.name
            print(f"[TRACE] Model requested tool: {name}")
            try:
                args = json.loads(call.function.arguments)
                if not isinstance(args, dict):
                    raise ValueError("Tool arguments must be a JSON object")
                if name not in TOOL_REGISTRY:
                    raise ValueError(f"Unknown tool: {name}")
                required = ("passenger_name", "from_city", "to_city", "travel_date")
                missing = [key for key in required if not args.get(key)]
                if missing:
                    result = {"status": "error", "message": f"Missing: {', '.join(missing)}. Ask the user."}
                else:
                    result = TOOL_REGISTRY[name](**args)
            except Exception as exc:
                result = {"status": "error", "message": str(exc)}
            print("[TRACE] Tool result:", result)
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "name": name,
                "content": json.dumps(result),
            })
    return "The agent reached its tool-call limit. Please try again with full details."


In [10]:
messages = new_conversation()
print("AGENT:", travel_agent("I want to book a flight.", messages))

AGENT: Sure, I can help with that. Could you please provide me with your full legal name, the city you'll be departing from, the city you're traveling to, and a future travel date in YYYY-MM-DD format?


In [11]:
print("AGENT:", travel_agent("My name is Alex Morgan. I am flying from London to Paris.", messages))

[TRACE] Model requested tool: book_flight
[TRACE] Tool result: {'status': 'error', 'message': 'Travel date must not be in the past.'}
AGENT: It looks like the travel date you provided is in the past. Could you please provide a future travel date instead?


In [12]:
print("AGENT:", travel_agent("Travel date is 2030-06-15.", messages))
print("SIMULATED BOOKINGS:", len(BOOKING_LOG))

[TRACE] Model requested tool: book_flight
[TRACE] Tool result: {'status': 'simulated_only', 'passenger_name': 'Alex Morgan', 'from_city': 'London', 'to_city': 'Paris', 'travel_date': '2030-06-15', 'message': 'SIMULATION ONLY: no flight has actually been booked.'}
AGENT: SIMULATED booking: No flight has actually been booked. Your details are: Name: Alex Morgan, Departing from: London, Flying to: Paris, Traveling on: 2030-06-15.
SIMULATED BOOKINGS: 1


In [13]:
for idx, msg in enumerate(messages):
    print(idx, msg.get("role"),
          "tool_calls=" + str(bool(msg.get("tool_calls"))),
          "tool_call_id=" + str(msg.get("tool_call_id", "")))

0 system tool_calls=False tool_call_id=
1 user tool_calls=False tool_call_id=
2 assistant tool_calls=False tool_call_id=
3 user tool_calls=False tool_call_id=
4 assistant tool_calls=True tool_call_id=
5 tool tool_calls=False tool_call_id=call_7efm7hvr
6 assistant tool_calls=False tool_call_id=
7 user tool_calls=False tool_call_id=
8 assistant tool_calls=True tool_call_id=
9 tool tool_calls=False tool_call_id=call_jqxielr8
10 assistant tool_calls=False tool_call_id=


In [14]:
import ipywidgets as widgets
from IPython.display import display, clear_output

chat_messages = new_conversation()
chat_log = widgets.Output(layout={"border": "1px solid #999", "height": "330px", "overflow_y": "auto"})
text_input = widgets.Text(placeholder="e.g., I want a flight to Paris", description="You:",
                          layout=widgets.Layout(width="75%"))
send_button = widgets.Button(description="Send", button_style="primary")
reset_button = widgets.Button(description="Reset chat")

def send_message(_=None):
    user_text = text_input.value.strip()
    if not user_text:
        return
    text_input.value = ""
    with chat_log:
        print("YOU:", user_text)
        try:
            answer = travel_agent(user_text, chat_messages)
            print("AGENT:", answer, "\n")
        except Exception as exc:
            print("ERROR:", str(exc), "\n")

def reset_chat(_=None):
    chat_messages.clear()
    chat_messages.extend(new_conversation())
    with chat_log:
        clear_output()
        print("Chat reset. Simulated booking records remain in BOOKING_LOG.")

send_button.on_click(send_message)
reset_button.on_click(reset_chat)
display(widgets.HTML("<h3>AI Travel Agent — Simulated Flight Booking</h3>"))
display(chat_log, widgets.HBox([text_input, send_button, reset_button]))

HTML(value='<h3>AI Travel Agent — Simulated Flight Booking</h3>')

Output(layout=Layout(border='1px solid #999', height='330px', overflow_y='auto'))

In [18]:

def cancel_simulated_booking(booking_id: str):
    """
    Cancel an existing simulated flight booking.
    """

    for booking in BOOKING_LOG:
        if booking["booking_id"] == booking_id:

            if booking["status"] == "simulated_cancelled":
                return {
                    "success": False,
                    "message": f"Booking {booking_id} has already been cancelled."
                }

            booking["status"] = "simulated_cancelled"

            return {
                "success": True,
                "message": f"Booking {booking_id} has been successfully cancelled."
            }

    return {
        "success": False,
        "message": f"Booking reference {booking_id} was not found."
    }


In [20]:

import json

def execute_tool(function_name, arguments):

    if function_name == "book_flight":
        return book_flight(**arguments)

    elif function_name == "cancel_simulated_booking":
        return cancel_simulated_booking(
            booking_id=arguments["booking_id"]
        )

    else:
        return {
            "success": False,
            "message": f"Unknown tool: {function_name}"
        }


In [21]:
TOOL_REGISTRY["cancel_simulated_booking"] = cancel_simulated_booking

In [22]:

system_prompt = """
You are an intelligent Travel Agent.

You can help users:
1. Search for flights.
2. Book simulated flights.
3. Cancel existing simulated flight bookings.

When a user requests cancellation:
- Automatically use cancel_simulated_booking.
- Ask for the booking reference if it is missing.
- Never invent a booking reference.
- Inform the user if the booking was successfully cancelled.
- Inform the user if the booking was already cancelled.
- Inform the user if the booking reference does not exist.

Always select the appropriate tool based on the user's request.
"""


In [23]:
# Re-define tools to include both book_flight and cancel_simulated_booking
tools = [
    {
        "type": "function",
        "function": {
            "name": "book_flight",
            "description": (
                "Simulate a flight booking only after the passenger has supplied "
                "their full name, departure city, destination city and future travel date. "
                "This does NOT make a real reservation."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "passenger_name": {"type": "string", "description": "Passenger full legal name"},
                    "from_city": {"type": "string", "description": "Departure city"},
                    "to_city": {"type": "string", "description": "Destination city"},
                    "travel_date": {"type": "string", "description": "Travel date in YYYY-MM-DD format"},
                },
                "required": ["passenger_name", "from_city", "to_city", "travel_date"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "cancel_simulated_booking",
            "description": "Cancel an existing simulated flight booking using its booking reference.",
            "parameters": {
                "type": "object",
                "properties": {
                    "booking_id": {
                        "type": "string",
                        "description": "The booking reference, for example DEMO-001."
                    }
                },
                "required": ["booking_id"],
                "additionalProperties": False
            }
        }
    }
]

# Update the SYSTEM_PROMPT to match the new instructions
SYSTEM_PROMPT = system_prompt
print("Tools configured:", [t['function']['name'] for t in tools])

Tools configured: ['book_flight', 'cancel_simulated_booking']


In [24]:
# Let's modify the book_flight function to actually issue a simulated booking_id so we can test cancellation!
def book_flight_with_id(passenger_name: str, from_city: str, to_city: str, travel_date: str) -> dict:
    import uuid
    res = book_flight(passenger_name, from_city, to_city, travel_date)
    if res.get("status") == "simulated_only":
        b_id = f"DEMO-{uuid.uuid4().hex[:6].upper()}"
        res["booking_id"] = b_id
        # update the record in BOOKING_LOG with its id
        BOOKING_LOG[-1]["booking_id"] = b_id
    return res

# Override in the registry
TOOL_REGISTRY["book_flight"] = book_flight_with_id
print("TOOL_REGISTRY updated.")

TOOL_REGISTRY updated.
